# Import packages

In [1]:
source("utils/helpers.R")
library(yaml)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
✔ purrr     1.2.2     ✔ tidyr     1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ readr::col_factor() masks scales::col_factor()
✖ purrr::discard()    masks scales::discard()
✖ dplyr::filter()     masks stats::filter()
✖ dplyr::lag()        masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: grid

ComplexHeatmap version 2.26.1
Bioconductor page: http://bioconductor.org/packages/ComplexHeatmap/
Github page: https://github.com/jokergoo/ComplexHeatmap
Documentation: http://jokergoo.github.io/ComplexHeatmap-reference

If you use it in published research, please cite either one:
- Gu, Z. Complex Heatmap Visualization. iMeta 2022.
- Gu, Z. Complex heatmaps reve

# Directories

In [3]:
config <- read_yaml("../config.yaml")
data_dir <- config$results_dir
results_dir <- config$figure_4

[1] "/Users/jawadalaaedeen/Desktop/PhD/NMC/results/figure_4"

# Parameters

In [ ]:
MIN_CELLS <- 10

# Importing the required data

In [5]:
celltype_metadata <- read_csv(
  paste0(data_dir, "/celltype_metadata_final.csv")
) %>%
  mutate(is_lung = ifelse(tumor_site == "Lung", "Lung", "Head and Neck/Extrahepatic")) %>%
  mutate(patient_num = as.numeric(str_extract(patient_exp, "(?<=NUT_)\\d+")),
         roi_num = as.numeric(str_extract(patient_exp, "(?<=ROI)\\d+"))) %>%
  arrange(patient_num, roi_num) %>%
  select(-patient_num, -roi_num)
celltype_metadata$patient_exp <- factor(celltype_metadata$patient_exp, levels = unique(celltype_metadata$patient_exp))
celltype_metadata$patient_ID <- factor(celltype_metadata$patient_ID, levels = unique(celltype_metadata$patient_ID))
patient_ID_order <- unique(celltype_metadata$patient_ID)
patient_exp_order <- unique(celltype_metadata$patient_exp)


myeloid <-  c("M1-like macrophages", "M2-like macrophages", "M1/M2-like macrophages", "DCs", "Granulocytes", "MDSCs", "Mast cells")
lymphoid <- c("CD4+ T cells", "CD8+ T cells", "Treg T cells", "NK cells", "B cells", "PCs")
nonimmune <- c("Actin+ cells", "Endothelial cells", "Lymphatic endothelial cells", "Epithelial cells")

celltype_metadata <- celltype_metadata %>%
  mutate(
    cell_category_simplified = case_when(
      cell_category %in% myeloid ~ "Myeloid",
      cell_category %in% lymphoid ~ "Lymphoid",
      cell_category %in% nonimmune ~ "Non-immune",
      cell_category == "NFC" ~ "NFC",
      cell_category == "Tumor cells" ~ "Tumor"
    ),
    cell_category_simplified = factor(
      cell_category_simplified,
      levels = c("NFC", "Non-immune", "Lymphoid", "Myeloid", "Tumor")
    )
  ) 

Rows: 553331 Columns: 17
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (10): cell_type, cell_category, patient_ID, exp_name, patient_exp, tumor...
dbl  (7): cell_id, Cell Center X, Cell Center Y, run, rois, survival_time_mo...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [6]:
survival_df <- celltype_metadata %>%
    select(patient_ID, tumor_site, survival_time_months) %>%
    distinct() %>%
    mutate(event = ifelse(patient_ID == "NUT_1", 0, 1),
        tumor_site = factor(tumor_site, levels = c("Lung", "Head and Neck", "Abdomen")), 
    patient_ID = factor(patient_ID, levels = patient_ID_order),
    is_lung = ifelse(tumor_site == "Lung", "Lung", "Head and Neck/Abdomen"),
        event = as.numeric(event),
        survival_time_months = as.numeric(survival_time_months)
      )

survival_df_order <- survival_df %>%
arrange(tumor_site, survival_time_months) %>%
mutate(patient_ID = factor(patient_ID, levels = unique(patient_ID)))

patient_ID_survival_order <-survival_df_order %>%
pull(patient_ID)

In [7]:
#niches df 
niches_df <- read.csv(file.path(data_dir, "niches_df.csv"))

In [9]:
surv_patient <- niches_df %>%
  distinct(patient_ID, survival_time_months) %>%
  left_join(survival_df %>% distinct(patient_ID, event), by = "patient_ID")

specs <- list(
  list(l="CD8+ T cells abundance",              d=pct_patient(niches_df,"CD8+ T cells"),                          s="median",   type="Non-spatial"),
  list(l="CD4+ T cells abundance",              d=pct_patient(niches_df,"CD4+ T cells"),                          s="median",   type="Non-spatial"),
  list(l="Granulocyte abundance",              d=pct_patient(niches_df,"Granulocytes"),                          s="median",   type="Non-spatial"),
  list(l="MDSCs abundance",                     d=pct_patient(niches_df,"MDSCs"),                                 s="median",   type="Non-spatial"),
  list(l="M2 macrophage abundance",            d=pct_patient(niches_df,"M2 macrophages"),                        s="median",   type="Non-spatial"),
  list(l="TLS-like nbhd presence",            d=niche_pct_patient(niches_df,"TLS_like"),                        s="presence", t=1, type="Spatial"),
  list(l="Granulocytic-suppressive nbhd presence",     d=niche_pct_patient(niches_df,"Granulocytic_suppressive"),        s="presence", t=1, type="Spatial"),
  list(l="Granulocytes in Tumor-myeloid nbhd",d=pct_patient(niches_df,"Granulocytes","Tumor_myeloid_enriched"), s="median",   type="Spatial"),
  list(l="MDSCs in Tumor-myeloid nbhd",       d=pct_patient(niches_df,"MDSCs","Tumor_myeloid_enriched"),        s="median",   type="Spatial"),
  list(l="M2-like in Tumor-myeloid nbhd",          d=pct_patient(niches_df,"M2 macrophages","Tumor_myeloid_enriched"),s="median",  type="Spatial")
)

results <- map(specs, ~ make_km(.x$d, .x$l, split = .x$s, thresh = .x$t %||% 1))

summary_tbl <- map_dfr(results, "summary") %>%
  left_join(tibble(variable = map_chr(specs, "l"),
                   type     = map_chr(specs, "type")), by = "variable") %>%
  arrange(p.value)
print(summary_tbl, width = Inf)

#export summary_tbl as RDS object
saveRDS(summary_tbl,  paste0(data_dir, "/summary_tbl.rds"))

CD8+ T cells abundance                     HR=0.52 [0.16-1.74]  p=0.292  PH_p=0.578  (Low=6, High=7)
CD4+ T cells abundance                     HR=0.93 [0.29-2.97]  p=0.899  PH_p=0.366  (Low=6, High=7)
Granulocyte abundance                      HR=4.04 [1.01-16.10]  p=0.048  PH_p=0.735  (Low=6, High=7)
MDSCs abundance                            HR=0.93 [0.30-2.94]  p=0.907  PH_p=0.504  (Low=6, High=7)
M2 macrophage abundance                    HR=0.42 [0.13-1.43]  p=0.166  PH_p=0.903  (Low=6, High=7)
TLS-like nbhd presence                     HR=0.73 [0.23-2.31]  p=0.587  PH_p=0.204  (Absent=7, Present=6)
Granulocytic-suppressive nbhd presence     HR=6.08 [1.23-29.97]  p=0.027  PH_p=0.636  (Absent=5, Present=8)
Granulocytes in Tumor-myeloid nbhd         HR=4.60 [1.15-18.37]  p=0.031  PH_p=0.422  (Low=6, High=7)
MDSCs in Tumor-myeloid nbhd                HR=2.82 [0.71-11.14]  p=0.139  PH_p=0.349  (Low=6, High=7)
M2-like in Tumor-myeloid nbhd              HR=0.42 [0.13-1.43]  p=0.166  PH